## MERGE DEI DATASET 

In [ ]:
import os
from pathlib import Path

# Get the current notebook's directory
notebook_dir = Path().absolute()

# Get the parent directory of the current notebook's folder
parent_dir = notebook_dir.parent

# Change the working directory to the parent folder
os.chdir(parent_dir)

# Print to verify the change
print("New Working Directory:", os.getcwd())

In [ ]:
import pandas as pd

# === STEP 1: Caricamento dei dataset di tempi di percorrenza ===
df_ps = pd.read_csv("DATASET_DIST/FINAL/tempi_percorrenza_ps_nuovo.csv")
df_porti = pd.read_csv("DATASET_DIST/FINAL/tempi_percorrenza_porti.csv")
df_aeroporti = pd.read_csv("DATASET_DIST/FINAL/tempi_percorrenza_aeroporti.csv")
df_stazioni = pd.read_csv("DATASET_DIST/FINAL/tempi_percorrenza_stazioni.csv")
df_ps_ped = pd.read_csv("DATASET_DIST/FINAL/tempi_percorrenza_ps_ped.csv")

# === STEP 2: Pulizia e rinomina colonne ===

# Pronto soccorso adulti
df_ps = df_ps.rename(columns={
    '2ps_code': 'ps_code_2',
    '2ps_subcode': 'ps_subcode_2',
    '2ps_latitude': 'ps_latitude_2',
    '2ps_longitude': 'ps_longitude_2',
    'travel_time_2ps': 'time_travel_ps_2',
    'ps_code': 'ps_code_1',
    'ps_subcode': 'ps_subcode_1',
    'ps_latitude': 'ps_latitude_1',
    'ps_longitude': 'ps_longitude_1',
    'travel_time_ps': 'time_travel_ps_1'
})

# Porti
df_porti = df_porti.drop(columns=['grid_SIGLA_PROVINCIA'], errors='ignore')
df_porti = df_porti.rename(columns={'travel_time_min': 'travel_time_port'})

# Aeroporti
df_aeroporti = df_aeroporti.drop(columns=['grid_SIGLA_PROVINCIA'], errors='ignore')
df_aeroporti = df_aeroporti.rename(columns={'travel_time_min': 'travel_time_airport'})

# Stazioni
df_stazioni = df_stazioni.drop(columns=['grid_SIGLA_PROVINCIA'], errors='ignore')
df_stazioni = df_stazioni.rename(columns={
    'travel_time_min': 'travel_time_ts',
    'ps_class': 'ts_class'
})

# PS pediatrici
df_ps_ped = df_ps_ped.drop(columns=['grid_SIGLA_PROVINCIA'], errors='ignore')
df_ps_ped = df_ps_ped.rename(columns={
    'ps_code': 'ps_code_ped',
    'ps_subcode': 'ps_subcode_ped',
    'ps_latitude': 'ps_latitude_ped',
    'ps_longitude': 'ps_longitude_ped',
    'ps_prov': 'ps_prov_ped',
    'travel_time_min': 'travel_time_ps_ped'
})

# === STEP 3: Merge progressivo dei dataset ===
# Merge per coordinate lat/lon di griglia
merged = df_ps \
    .merge(df_aeroporti, on=['grid_latitude', 'grid_longitude']) \
    .merge(df_porti, on=['grid_latitude', 'grid_longitude']) \
    .merge(df_stazioni, on=['grid_latitude', 'grid_longitude']) \
    .merge(df_ps_ped, on=['grid_latitude', 'grid_longitude'])

# Rinomina coordinate
merged = merged.rename(columns={
    'grid_latitude': 'latitude',
    'grid_longitude': 'longitude'
})


# === STEP 5: aggiunta della popolazione ===

pop_df = pd.read_csv("DATASET_DIST/popolazione_redistribuita.csv")
pop_df = pop_df.rename(columns={
    'latitudine': 'latitude',
    'longitudine': 'longitude'
})

merged = pd.merge(merged, pop_df, on=['latitude', 'longitude'], how='left')

# === STEP 6: Salvataggio ===
merged.to_csv("DATASET_DIST/total_accessibility.csv", index=False)

print("✅ Dataset unificato salvato correttamente!")


In [ ]:
import pandas as pd

# Colonne da eliminare
colonne_da_droppare = [
    "TOT_P_2021", "M", "F", "Y_LT15", "Y1564", "Y_GE65",
    "NAT", "EU_OTH", "OTH", "EMP", "SAME", "CHG_IN", "CHG_OUT"
]

# Carica il file con encoding compatibile
df = pd.read_csv("DATASET_DIST/total_accessibility.csv", encoding="latin1")

# Elimina le colonne se presenti
df = df.drop(columns=[col for col in colonne_da_droppare if col in df.columns])

# Salva il file pulito
df.to_csv("DATASET_DIST/total_accessibility_cleaned.csv", index=False)

print("✅ File CSV pulito salvato come total_accessibility_cleaned.csv")


##### MERGE AGGIUNGENDO LA POPOLAZIONE (PROBLEMI DI ARROTONDAMENTO RISOLTI ALLA OTTAVA CIFRA DECIMALE)

In [ ]:
import pandas as pd

# Carica i due file
df_cleaned = pd.read_csv("DATASET_DIST/total_accessibility_cleaned.csv")
df_pop = pd.read_csv("DATASET_DIST/popolazione_redistribuita.csv")

# Rinominare le colonne nel file popolazione per uniformità
df_pop = df_pop.rename(columns={
    'latitudine': 'latitude',
    'longitudine': 'longitude'
})

# Arrotondamento delle coordinate alla 8ª cifra decimale
df_cleaned['latitude_rounded'] = df_cleaned['latitude'].round(7)
df_cleaned['longitude_rounded'] = df_cleaned['longitude'].round(7)

df_pop['latitude_rounded'] = df_pop['latitude'].round(7)
df_pop['longitude_rounded'] = df_pop['longitude'].round(7)

# Merge su latitudine e longitudine arrotondate
df_merged = pd.merge(df_cleaned, df_pop, on=["latitude_rounded", "longitude_rounded"], how="left")

# Rimuovi le colonne di coordinate arrotondate (facoltativo)
df_merged = df_merged.drop(['latitude_rounded', 'longitude_rounded'], axis=1)

# Salva il file unificato
df_merged.to_csv("DATASET_DIST/total_accessibility_enriched.csv", index=False)

print("✅ Merge completato: salvato come total_accessibility_enriched.csv")


In [ ]:
print(f"Numero di righe nel file CSV: {len(df_ps)}")
print(f"Numero di righe nel file CSV: {len(df_ps_ped)}")
print(f"Numero di righe nel file CSV: {len(df_aeroporti)}")
print(f"Numero di righe nel file CSV: {len(df_porti)}")
print(f"Numero di righe nel file CSV: {len(df_stazioni)}")
print(f"Numero di righe nel file CSV: {len(df_merged)}")
print(f"Numero di righe nel file CSV: {len(df)}")

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Carica il file CSV finale
df_merged = pd.read_csv("DATASET_DIST/total_accessibility_enriched.csv")

# Crea una geometria da latitudine e longitudine
geometry = [Point(xy) for xy in zip(df_merged['longitude_x'], df_merged['latitude_x'])]

# Crea un GeoDataFrame
gdf = gpd.GeoDataFrame(df_merged, geometry=geometry)

# Imposta il sistema di riferimento spaziale (SRID).
gdf.set_crs('EPSG:4326', allow_override=True, inplace=True)

# Salva il GeoDataFrame come file GPKG
gdf.to_file("DATASET_DIST/total_accessibility_enriched.gpkg", driver="GPKG")

print("✅ Conversione completata: file salvato come total_accessibility_enriched.gpkg")


In [ ]:
import geopandas as gpd

# Carica il file GPKG
gdf_gpkg = gpd.read_file("DATASET_DIST/total_accessibility_enriched.gpkg")

# Verifica il numero di righe (dovrebbe corrispondere al CSV finale)
print(f"Numero di righe nel file GPKG: {len(gdf_gpkg)}")

# Carica il CSV finale
df_csv = pd.read_csv("DATASET_DIST/total_accessibility_enriched.csv")
print(f"Numero di righe nel file CSV: {len(df_csv)}")

# Verifica se il numero di righe è lo stesso
if len(gdf_gpkg) == len(df_csv):
    print("Le righe del CSV e del GPKG coincidono.")
else:
    print("Attenzione: il numero di righe non coincide!")

# Verifica che tutte le geometrie siano valide
invalid_geometries = gdf_gpkg[~gdf_gpkg.is_valid]
if len(invalid_geometries) > 0:
    print(f"Ci sono {len(invalid_geometries)} righe con geometrie non valide.")
else:
    print("Tutte le geometrie sono valide.")

# Mostra un esempio di geometria per confermare che i punti siano corretti
print("Esempio di geometrie (coordinate):")
print(gdf_gpkg.geometry.head())


In [ ]:
print(f"Numero di righe nel CSV originale: {len(df)}")
print(f"Numero di righe nel GPKG: {len(gdf)}")


In [ ]:
import pandas as pd
import geopandas as gpd

# Carica il CSV
csv_file = 'DATASET_DIST/total_accessibility.csv'
df = pd.read_csv(csv_file)

# Rimuovi le colonne specificate se esistono
cols_to_remove = ['COD_MACRO_x', 'latitude_y', 'longitude_y']
df = df.drop(columns=[col for col in cols_to_remove if col in df.columns])

# Salva il nuovo CSV
df.to_csv('DATASET_DIST/total_accessibility_clean.csv', index=False)

# Carica il GPKG
gpkg_file = 'DATASET_DIST/total_accessibility.gpkg'
gdf = gpd.read_file(gpkg_file)

# Rimuovi le colonne specificate se esistono
gdf = gdf.drop(columns=[col for col in cols_to_remove if col in gdf.columns])

# Salva il nuovo GPKG
gdf.to_file('DATASET_DIST/total_accessibility_clean.gpkg', driver='GPKG')
